In [7]:
from gensim.models import Word2Vec
import numpy as np
import re

# =========================
# 1) 
# =========================
def normalize_ar(text: str) -> str:
    text = text.lower()
    # إبقاء العربية + الأرقام + المسافات
    text = re.sub(r"[^\u0600-\u06FF0-9\s]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_to_sentences(text: str):
    # تقسيم بسيط للجمل (يمكن تحسينه لاحقًا)
    parts = re.split(r"[.\n!?؟،؛]+", text)
    parts = [p.strip() for p in parts if p.strip()]
    return parts

def tokenize(text: str):
    return normalize_ar(text).split()

# =========================
# 2) قراءة dataset.txt وتحويله لقائمة جمل (كل جملة قائمة كلمات)
# =========================
with open("dataset1.txt", "r", encoding="utf-8") as f:
    raw = f.read()

sentences = [tokenize(s) for s in split_to_sentences(raw)]
sentences


[[],
 ['جيرالد', 'هانلي', 'يتيمه', 'تاريخ', 'اكتوبر'],
 ['صندوق', 'معلومات', 'شخص'],
 ['جيرالد', 'هانلي', 'انج', 'فبراير', 'سبتمبر'],
 ['روائي', 'ايرلندي', 'فبراير'],
 ['تحديد'],
 ['المصادر'],
 ['مراجع'],
 ['شريط', 'بوابات', 'اعلام', 'ادب', 'ايرلندا'],
 ['بذره', 'كاتب', 'ايرلندي'],
 ['ضبط', 'استنادي'],
 ['تصنيف', 'روائيون', 'بريطانيون', 'في', 'القرن'],
 ['تصنيف', 'روائيون', 'من', 'ليفربول'],
 ['تصنيف', 'كتاب', 'ادب', 'رحلات', 'بريطانيون'],
 ['تصنيف', 'كتاب', 'من', 'ليفربول'],
 ['تصنيف', 'مواليد'],
 ['تصنيف', 'وفيات', 'أدب'],
 ['كمال', 'قرور', 'كاتب', 'يتيمه', 'تاريخ', 'يونيو'],
 ['صندوق', 'معلومات', 'كاتب'],
 ['المدرسه', 'الام', 'جامعه', 'قسنطينه', 'منتوري', 'جامعه', 'قسنطينه'],
 ['الاسم', 'الادبي'],
 ['جوائز'],
 ['اسم', 'الميلاد'],
 ['مكان', 'الميلاد', 'بني', 'عزيز', 'ولايه', 'سطيف', 'الجزائر'],
 ['اطفال'],
 ['المواطنه'],
 ['سبب', 'الوفاه'],
 ['مكان', 'الوفاه'],
 ['الدراسه', 'ليسانس', 'معهد', 'الاداب', 'واللغه', 'العربيه'],
 ['الاثنيه'],
 ['النوع', 'صحافه', 'وقصص', 'قصيره', 'روايه', '

In [9]:
sentences = [s for s in sentences if len(s) >= 3]  # تجاهل الجمل القصيرة جدًا

print("عدد الجمل:", len(sentences))
print("مثال جملة:", sentences[0][:10] if sentences else "لا يوجد")



عدد الجمل: 20553
مثال جملة: ['جيرالد', 'هانلي', 'يتيمه', 'تاريخ', 'اكتوبر']


In [11]:

# =========================
# 3) تدريب Word2Vec (Skip-gram)
# =========================
# sg=1 => Skip-gram
# window=2
# vector_size => حجم الشعاع
# min_count => تجاهل الكلمات النادرة (خليه 1 للتجارب)
# workers => عدد الأنوية
model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=2,
    min_count=1,
    sg=1,
    negative=10,
    epochs=10,
    workers=10
)

In [13]:
# =========================
# 4) شعاع كلمة + أقرب كلمات
# =========================
def word_vector(word: str):
    w = normalize_ar(word)
    if w in model.wv:
        return model.wv[w]
    return None


In [22]:
word_vector("عزيز")

array([ 0.00364307,  0.1216119 ,  0.1716979 ,  0.06080609,  0.0609096 ,
       -0.11916237, -0.16219117,  0.26387796,  0.14224525, -0.17972748,
       -0.21335368, -0.29513046, -0.01502741,  0.13660192,  0.02483036,
       -0.23977317, -0.08953559, -0.24053368,  0.12679157, -0.3088611 ,
        0.25162336,  0.12755503,  0.08413032, -0.10369055,  0.04880738,
        0.01345758, -0.24894597,  0.15228245, -0.21593782,  0.06861886,
        0.09639045, -0.12949823, -0.08773229, -0.18277213, -0.05824156,
        0.10044359,  0.0739193 , -0.1786156 , -0.16796379, -0.23717032,
       -0.05581419, -0.13760059, -0.28662983,  0.01100753,  0.3241531 ,
       -0.16122605, -0.12773423, -0.10670691,  0.1101189 ,  0.23143135,
       -0.08842216, -0.13112666, -0.2884545 , -0.29892096, -0.02888757,
       -0.02377657,  0.06733565, -0.02995298, -0.07630518,  0.0541101 ,
       -0.06941763,  0.22189315, -0.0022687 , -0.07719549, -0.24910888,
        0.22729115,  0.15736598,  0.07198245, -0.15976103,  0.14

In [29]:

def most_similar(word: str, topn: int = 10):
    w = normalize_ar(word)
    if w not in model.wv:
        return []
    return model.wv.most_similar(w, topn=topn)

print("\nأقرب كلمات لـ (الولايات):")
print(most_similar("خلال",10))



أقرب كلمات لـ (الولايات):
[('اثناء', 0.8036854267120361), ('بدات', 0.7860570549964905), ('قبل', 0.7849846482276917), ('فان', 0.7818092107772827), ('اجل', 0.7817336916923523), ('الاخيره', 0.7806265950202942), ('طوال', 0.7775583267211914), ('اشهر', 0.7757300734519958), ('الفنيه', 0.7638091444969177), ('منفيو', 0.7620440125465393)]


In [33]:

# =========================
# 5) شعاع مقالة كاملة (Average of word vectors)
# =========================
def article_vector(text: str, method: str = "mean"):
    toks = tokenize(text)
    vecs = [model.wv[w] for w in toks if w in model.wv]
    if not vecs:
        return None, 0

    vecs = np.array(vecs, dtype=np.float32)
    if method == "sum":
        return vecs.sum(axis=0), len(vecs)
    return vecs.mean(axis=0), len(vecs)

article = "أعلنت الولايات المتحدة فرض حزمة جديدة  من العقوبات على قطاع النفط"
v, used = article_vector(article, method="mean")
print("\nعدد الكلمات المستخدمة لبناء شعاع المقالة:", used)
print("أول 10 قيم من شعاع المقالة:", None if v is None else v[:10])
v


عدد الكلمات المستخدمة لبناء شعاع المقالة: 7
أول 10 قيم من شعاع المقالة: [ 0.09820397  0.05608604  0.04794088  0.03158684  0.26541558 -0.16097549
 -0.14236279  0.17574994  0.24498214 -0.05948274]


array([ 0.09820397,  0.05608604,  0.04794088,  0.03158684,  0.26541558,
       -0.16097549, -0.14236279,  0.17574994,  0.24498214, -0.05948274,
       -0.16658738, -0.363961  , -0.0409479 ,  0.12331523, -0.03488668,
       -0.24093316,  0.02016042, -0.10192831,  0.03904407, -0.36357138,
        0.16406557,  0.09399509, -0.02535763, -0.06815264,  0.15092719,
       -0.20875685, -0.30005035,  0.16903101, -0.3349743 ,  0.12844639,
        0.22037028, -0.3566111 , -0.07198387, -0.22748394, -0.14697964,
        0.04467881,  0.05073998, -0.10106429, -0.08516049, -0.42187697,
        0.10724147, -0.17890082, -0.2851099 , -0.09402962,  0.41512737,
       -0.13949183, -0.3428219 , -0.0059127 ,  0.16363896,  0.42112353,
        0.00331602, -0.21219662, -0.30403504, -0.15722787, -0.14933585,
        0.05345533,  0.19136223, -0.11760221, -0.2728985 ,  0.07629762,
       -0.05720558,  0.08943778,  0.22174509, -0.00495941, -0.33460793,
        0.4546434 ,  0.19306503,  0.12577285, -0.19987549,  0.21

In [35]:

# =========================
# 6) (اختياري) حفظ النموذج والمتجهات
# =========================
model.save("w2v_ar.model")
model.wv.save_word2vec_format("w2v_ar.vec", binary=False)
